# Speculative Decoding: MMLU Benchmark Experiment

This notebook runs the MMLU (Massive Multitask Language Understanding) benchmark to evaluate speculative decoding performance.

## What is Speculative Decoding?

Speculative decoding accelerates LLM inference by using a small "draft" model to propose tokens, which are then verified by a larger "target" model in a single forward pass. Tokens are accepted with probability `min(1, p(x)/q(x))`, where:
- `p(x)` = target model probability
- `q(x)` = draft model probability

This produces **exact samples from the target distribution** while being significantly faster.

## Experiment Setup

- **Draft model**: `HuggingFaceTB/SmolLM-135M` (135M parameters)
- **Target model**: `HuggingFaceTB/SmolLM2-1.7B` (1.7B parameters)
- **Dataset**: MMLU (57 multiple-choice tasks)
- **Decoding**: Greedy (temperature=0) for deterministic evaluation

## GPU Requirement

This notebook requires a GPU runtime. Go to **Runtime → Change runtime type → T4 GPU**.

## 1. Setup: Clone Repository and Install Dependencies

First, we clone the repository and install all required packages. This includes:
- `torch` — PyTorch for tensor operations
- `transformers` — HuggingFace model loading
- `datasets` — HuggingFace dataset loading (for MMLU)
- `pyyaml` — Config file parsing

In [ ]:
# Clone the repository
!git clone https://github.com/Yahtze/speculative-decoding.git
%cd speculative-decoding

In [ ]:
# Install dependencies
!pip install -e .

## 2. Verify GPU Availability

Confirm that we have access to a GPU. The code auto-detects CUDA and places models on GPU automatically.

Expected output: `CUDA available: True` with GPU name (e.g., Tesla T4)

In [ ]:
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. Go to Runtime → Change runtime type → T4 GPU")

## 3. Load Configuration

We load the experiment configuration from `config.yml`. This controls:
- Which MMLU task to evaluate
- Model names (draft and target)
- Decoding parameters (k, temperature, top_k)
- Output settings

You can modify `task` to run different MMLU subjects (see full list below).

In [ ]:
import yaml
from pathlib import Path

# Load config
config_path = Path("speculative-decoding/experiments/mmlu_speculative_decoding/config.yml")
with open(config_path) as f:
    config = yaml.safe_load(f)

# Display current config
print("Configuration:")
print(f"  Task:           {config['mmlu']['task']}")
print(f"  Draft model:    {config['models']['draft']['name']}")
print(f"  Target model:   {config['models']['target']['name']}")
print(f"  k:              {config['decoding']['k']}")
print(f"  Temperature:    {config['decoding']['temperature']}")
print(f"  Max new tokens: {config['decoding']['max_new_tokens']}")

# Available MMLU tasks
MMLU_TASKS = [
    'abstract_algebra', 'anatomy', 'astronomy', 'business_ethics', 'clinical_knowledge',
    'college_biology', 'college_chemistry', 'college_computer_science', 'college_mathematics',
    'college_medicine', 'college_physics', 'computer_security', 'conceptual_physics',
    'econometrics', 'electrical_engineering', 'elementary_mathematics', 'formal_logic',
    'global_facts', 'high_school_biology', 'high_school_chemistry', 'high_school_computer_science',
    'high_school_european_history', 'high_school_geography', 'high_school_government_and_politics',
    'high_school_macroeconomics', 'high_school_mathematics', 'high_school_microeconomics',
    'high_school_physics', 'high_school_psychology', 'high_school_statistics',
    'high_school_us_history', 'high_school_world_history', 'human_aging', 'human_sexuality',
    'international_law', 'jurisprudence', 'logical_fallacies', 'machine_learning', 'management',
    'marketing', 'medical_genetics', 'miscellaneous', 'moral_disputes', 'moral_scenarios',
    'nutrition', 'philosophy', 'prehistory', 'professional_accounting', 'professional_law',
    'professional_medicine', 'professional_psychology', 'public_relations', 'security_studies',
    'sociology', 'us_foreign_policy', 'virology', 'world_religions'
]
print(f"\nAvailable tasks ({len(MMLU_TASKS)}): {', '.join(MMLU_TASKS[:5])}...")

### Optional: Change the MMLU Task

Uncomment and modify the cell below to run a different MMLU task. Common choices:
- `anatomy` — Medical anatomy questions
- `abstract_algebra` — Math/algebra
- `computer_security` — Cybersecurity
- `machine_learning` — ML concepts

In [ ]:
# Uncomment to override the task:
# config['mmlu']['task'] = 'computer_security'

## 4. Load MMLU Dataset

We load the MMLU dataset from HuggingFace. Each example contains:
- `question` — The question text
- `choices` — List of 4 multiple-choice options (A, B, C, D)
- `answer` — Index of correct answer (0-3)

The questions are formatted as:
```
Question: What is the capital of France?
A. London
B. Paris
C. Berlin
D. Madrid
Answer:
```

In [ ]:
from datasets import load_dataset

CHOICE_LABELS = ["A", "B", "C", "D"]

def format_mmlu_prompt(question: str, choices: list) -> str:
    """Format an MMLU question as a multiple-choice prompt."""
    choices_str = "\n".join(
        f"{label}. {choice}" for label, choice in zip(CHOICE_LABELS, choices)
    )
    return f"Question: {question}\n{choices_str}\nAnswer:"

# Load dataset
task = config['mmlu']['task']
dataset_name = config['mmlu']['dataset']
max_questions = config['mmlu'].get('max_questions')

print(f"Loading MMLU task: {task}")
ds = load_dataset(dataset_name, task, split="test")

if max_questions:
    ds = ds.select(range(min(max_questions, len(ds))))

# Format prompts
prompts = []
answer_indices = []

for example in ds:
    prompts.append(format_mmlu_prompt(example["question"], example["choices"]))
    answer_indices.append(example["answer"])

print(f"Loaded {len(prompts)} questions")
print(f"\nExample prompt:\n{prompts[0]}")

## 5. Load Models

We load both models:
- **Draft model** (135M params): Small, fast model that proposes candidate tokens
- **Target model** (1.7B params): Large, accurate model that verifies candidates

Models are automatically placed on GPU if available. This may take 1-2 minutes to download and load weights.

In [ ]:
import sys
sys.path.insert(0, "src")

from models.wrapper import LanguageModel

draft_model_name = config['models']['draft']['name']
target_model_name = config['models']['target']['name']

print(f"Loading draft model: {draft_model_name}")
draft_model = LanguageModel(draft_model_name)
print(f"  Device: {draft_model.device}")
print(f"  Parameters: {draft_model.num_parameters / 1e6:.1f}M")

print(f"\nLoading target model: {target_model_name}")
target_model = LanguageModel(target_model_name)
print(f"  Device: {target_model.device}")
print(f"  Parameters: {target_model.num_parameters / 1e6:.1f}M")

## 6. Run Speculative Decoding Experiment

Now we run the main experiment. For each question:
1. Draft model proposes K tokens via top-k sampling (or greedy when temperature=0)
2. Target model verifies all K tokens in one forward pass
3. Tokens accepted if draft matches target's argmax (greedy mode)
4. Rejected tokens resampled from target distribution

We track:
- **Acceptance rate**: How often draft model agrees with target
- **Throughput**: Tokens generated per second
- **Latency**: Time per question
- **Accuracy**: Correct answers on MMLU

In [ ]:
from decoding.speculative import SpeculativeDecoder
from benchmarks.runner import run_experiment, progress_printer

# Get decoding params
k = config['decoding']['k']
max_new_tokens = config['decoding']['max_new_tokens']
temperature = config['decoding']['temperature']
top_k = config['decoding']['top_k']

print(f"Running speculative decoding:")
print(f"  k={k}, temperature={temperature}, top_k={top_k}")
print(f"  Max new tokens: {max_new_tokens}")
print(f"  Questions: {len(prompts)}")
print()

# Run experiment
result = run_experiment(
    draft_model=draft_model,
    target_model=target_model,
    k=k,
    prompts=prompts,
    max_new_tokens=max_new_tokens,
    temperature=temperature,
    top_k=top_k,
    progress_fn=progress_printer,
)

## 7. Evaluate Accuracy

We extract the model's answer from the generated text and compare against ground truth.

The model should output a letter (A, B, C, or D) after "Answer:". We look for the first valid letter in the response.

In [ ]:
def extract_answer(generated_text: str, prompt: str) -> str | None:
    """Extract the answer letter from generated text."""
    response = generated_text[len(prompt):].strip()
    if not response:
        return None
    for char in response[:10]:  # Check first 10 chars
        if char.upper() in CHOICE_LABELS:
            return char.upper()
    return None

# Calculate accuracy
correct = 0
invalid = 0

for i, (pr, true_idx) in enumerate(zip(result.prompt_results, answer_indices)):
    predicted = extract_answer(pr.output, pr.prompt)
    if predicted is None:
        invalid += 1
    else:
        if CHOICE_LABELS.index(predicted) == true_idx:
            correct += 1

total = len(result.prompt_results)
accuracy = correct / total if total > 0 else 0.0

print(f"Accuracy: {accuracy:.1%} ({correct}/{total})")
print(f"Invalid format: {invalid}")

## 8. Display Results Summary

Full experiment results including:
- Acceptance rate (how often draft matches target)
- Throughput (tokens/second)
- Latency (seconds/question)
- Total draft and target model calls

In [ ]:
print(result.summary())
print(f"MMLU Accuracy: {accuracy:.1%}")

## 9. Save Results

Results are saved to `speculative-decoding/results/mmlu_speculative_decoding/`:
- `mmlu_{task}.json` — Full results with per-question details
- `mmlu_{task}.csv` — Per-question results in CSV format

In [ ]:
import json
from pathlib import Path

results_dir = Path("speculative-decoding/results/mmlu_speculative_decoding")
results_dir.mkdir(parents=True, exist_ok=True)

save_name = f"mmlu_{task}"

# Save JSON
json_path = results_dir / f"{save_name}.json"
data = {
    "config": {
        "task": task,
        "draft_model": draft_model_name,
        "target_model": target_model_name,
        "k": k,
        "temperature": temperature,
        "max_new_tokens": max_new_tokens,
    },
    "metrics": {
        "accuracy": accuracy,
        "acceptance_rate": result.acceptance_rate,
        "throughput": result.throughput,
        "latency": result.latency,
    },
    "prompts": [
        {
            "prompt": pr.prompt,
            "output": pr.output,
            "accepted": pr.stats.accepted_tokens,
            "drafted": pr.stats.drafted_tokens,
        }
        for pr in result.prompt_results
    ],
}

with open(json_path, "w") as f:
    json.dump(data, f, indent=2)

print(f"Results saved to: {json_path}")

## 10. View Sample Outputs

Look at a few example questions to see what the model generated.

In [ ]:
# Show first 3 examples
for i in range(min(3, len(result.prompt_results))):
    pr = result.prompt_results[i]
    true_answer = CHOICE_LABELS[answer_indices[i]]
    predicted = extract_answer(pr.output, pr.prompt)
    
    print(f"\n{'='*60}")
    print(f"Question {i+1}:")
    print(pr.prompt[:200] + "...")
    print(f"\nModel output: {pr.output[len(pr.prompt):].strip()[:50]}")
    print(f"Predicted: {predicted} | Correct: {true_answer} | {'✓' if predicted == true_answer else '✗'}")
    print(f"Acceptance rate: {pr.stats.acceptance_rate:.1%}")

## Next Steps

To run more experiments:

1. **Different tasks**: Change `config['mmlu']['task']` to any of the 57 MMLU subjects
2. **Different k values**: Modify `config['decoding']['k']` to test k=1,3,5,10
3. **Baseline comparison**: Run `experiments/mmlu_baseline/run_mmlu_baseline.py` to compare with standard greedy decoding

### Compare with Baseline

To see the speedup from speculative decoding, run the baseline experiment that uses only the target model with greedy decoding.

In [ ]:
# Optional: Run baseline for comparison
# !python speculative-decoding/experiments/mmlu_baseline/run_mmlu_baseline.py --task {task}